In [4]:
from pathlib import Path

In [5]:
gt_directory = Path("video/gt")
log_directory = Path("video/logs")

In [6]:
def calculate_tiou(seg1, seg2):
    """Calculates temporal Intersection over Union between two segments."""
    s1, e1 = seg1
    s2, e2 = seg2
    intersection = max(0, min(e1, e2) - max(s1, s2))
    union = (e1 - s1) + (e2 - s2) - intersection
    return intersection / union if union > 0 else 0.0

def load_events(filepath):
    """Loads interaction events from a CSV file into a dictionary."""
    events = {}
    with filepath.open('r') as f:
        for line in f:
            if not line.strip(): 
                continue
            cols = line.strip().split(',')
            
            # Extract key (Human ID, Object ID, Class) and segment (Start Frame, End Frame)
            key = (cols[0], cols[1], cols[2])
            seg = (int(cols[3]), int(cols[4]))
            
            if key not in events:
                events[key] = []
            events[key].append(seg)
    return events

if not gt_directory.exists() or not log_directory.exists():
    print("Error: Ground truth or logs directory not found.")
else:
    # Iterate through all ground truth CSVs
    for gt_file in gt_directory.glob("*_gt.csv"):
        
        # Extract the base [name] by removing the '_gt.csv' suffix
        video_name = gt_file.name.replace("_gt.csv", "")
        
        # Construct the path for the expected matching log file
        log_file = log_directory / f"{video_name}_interactions_summary.csv"
        
        if not log_file.exists():
            print(f"\nSkipping {video_name}: No matching log file found at {log_file}")
            continue
            
        print(f"\n=== Evaluating Video: {video_name} ===")
        
        gt_events = load_events(gt_file)
        log_events = load_events(log_file)
        
        # Compare logs against ground truth
        for key in log_events:
            if key in gt_events:
                for log_seg in log_events[key]:
                    best_tiou = 0.0
                    best_gt_seg = None
                    
                    # Find the ground truth segment that overlaps the most
                    for gt_seg in gt_events[key]:
                        tiou = calculate_tiou(log_seg, gt_seg)
                        if tiou > best_tiou:
                            best_tiou = tiou
                            best_gt_seg = gt_seg
                    
                    h, o, c = key
                    print(f"ID({h},{o}) Class:{c} | Log:{log_seg} matched GT:{best_gt_seg} | tIoU: {best_tiou:.3f}")
            else:
                h, o, c = key
                print(f"ID({h},{o}) Class:{c} | Log:{log_seg} | tIoU: 0.000 (Not in Ground Truth)")


=== Evaluating Video: vid1 ===
ID(1,2) Class:phone | Log:(355, 3915) matched GT:(355, 3915) | tIoU: 1.000
